# Final Model — XGBoost + LightGBM + CNN Ensemble (Late Fusion)
**DS340 — Multiclass Genre Classification**

**Architecture:**
- Stage 1A: XGBoost on 32 engineered Spotify tabular features
- Stage 1B: LightGBM on 32 engineered Spotify tabular features
- Stage 1C: 1D CNN on 59 MFCC features
- **Late Fusion**: XGB 60% + LGB 30% + CNN 10% → parent genre prediction

**Data:** 9 parent genres covered by MFCC extraction pipeline (Rock, Metal, Electronic, Latin, Jazz/Blues, Classical/Instrumental, Country/Folk, Reggae, World/Other). Deduplicated to one genre label per track.

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'XGBoost: {xgb.__version__}, LightGBM: {lgb.__version__}, PyTorch: {torch.__version__}')

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Device: cpu
XGBoost: 3.2.0, LightGBM: 4.6.0, PyTorch: 2.2.2


---
## 1. Load Spotify Tabular Data (Stage 1A — XGBoost)

In [2]:
# 9 parent genres covered by our MFCC extraction pipeline
# (Hip-Hop/R&B, House/Dance, Pop excluded — no MFCC clips extracted for those)
MFCC_GENRES = ['Rock','Metal','Electronic','Latin','Jazz/Blues',
               'Classical/Instrumental','Country/Folk','Reggae','World/Other']

GENRE_MAP = {
    'rock':'Rock','alt-rock':'Rock','alternative':'Rock','indie':'Rock','grunge':'Rock',
    'punk':'Rock','punk-rock':'Rock','psych-rock':'Rock','rockabilly':'Rock','rock-n-roll':'Rock','hard-rock':'Rock',
    'metal':'Metal','black-metal':'Metal','death-metal':'Metal','heavy-metal':'Metal','metalcore':'Metal','grindcore':'Metal',
    'electronic':'Electronic','electro':'Electronic','ambient':'Electronic','idm':'Electronic',
    'industrial':'Electronic','new-age':'Electronic','synth-pop':'Electronic',
    'latin':'Latin','salsa':'Latin','samba':'Latin','bossanova':'Latin','mpb':'Latin',
    'pagode':'Latin','sertanejo':'Latin','reggaeton':'Latin',
    'jazz':'Jazz/Blues','blues':'Jazz/Blues',
    'classical':'Classical/Instrumental','opera':'Classical/Instrumental','piano':'Classical/Instrumental',
    'guitar':'Classical/Instrumental','acoustic':'Classical/Instrumental',
    'country':'Country/Folk','folk':'Country/Folk','bluegrass':'Country/Folk','honky-tonk':'Country/Folk',
    'reggae':'Reggae','dub':'Reggae','ska':'Reggae',
    'world-music':'World/Other','afrobeat':'World/Other','indian':'World/Other','iranian':'World/Other',
    'turkish':'World/Other','forro':'World/Other','tango':'World/Other','disney':'World/Other',
    'children':'World/Other','comedy':'World/Other','gospel':'World/Other','sleep':'World/Other',
    'show-tunes':'World/Other','romance':'World/Other',
}

df = pd.read_csv('../Data/spotify-tracks-dataset.csv')
df['genre'] = df['track_genre'].str.lower().map(GENRE_MAP)
df = df[df['genre'].isin(MFCC_GENRES)].dropna(subset=['genre'])

# Deduplicate: keep first occurrence of each track_id so each song
# appears under exactly one genre label (removes 3,809 conflicting rows)
df = df.drop_duplicates(subset='track_id', keep='first').reset_index(drop=True)
df['explicit'] = df['explicit'].astype(int)

print(f'Dataset (9 genres, deduplicated): {df.shape[0]:,} rows')
print(df['genre'].value_counts())

Dataset (9 genres, deduplicated): 50,196 rows
genre
World/Other               13387
Rock                       7611
Electronic                 6694
Latin                      5354
Metal                      4920
Classical/Instrumental     4509
Country/Folk               3780
Reggae                     2208
Jazz/Blues                 1733
Name: count, dtype: int64


---
## 2. Feature Engineering (28 features)

In [3]:
def engineer_features(d):
    d = d.copy()
    d['energy_acoustic'] = d['energy'] * d['acousticness']
    d['speech_dance']    = d['speechiness'] * d['danceability']
    d['instru_acoustic'] = d['instrumentalness'] * d['acousticness']
    d['energy_valence']  = d['energy'] - d['valence']
    d['dance_valence']   = d['danceability'] * d['valence']
    d['loudness_norm']   = (d['loudness'] + 60) / 60
    d['loudness_sq']     = d['loudness_norm'] ** 2
    d['tempo_slow']      = (d['tempo'] < 90).astype(int)
    d['tempo_mid']       = ((d['tempo'] >= 90) & (d['tempo'] < 120)).astype(int)
    d['tempo_fast']      = ((d['tempo'] >= 120) & (d['tempo'] < 140)).astype(int)
    d['tempo_vfast']     = (d['tempo'] >= 140).astype(int)
    d['log_duration']    = np.log1p(d['duration_ms'])
    d['log_popularity']  = np.log1p(d['popularity'])
    d['energy_sq']       = d['energy'] ** 2
    d['acoustic_sq']     = d['acousticness'] ** 2
    d['loud_energy']     = d['loudness_norm'] * d['energy']
    d['energy_dance']    = d['energy'] * d['danceability']
    return d

BASE_COLS = ['popularity','duration_ms','explicit','danceability','energy','key','loudness','mode',
             'speechiness','acousticness','instrumentalness','liveness','valence','tempo','time_signature']
ENG_COLS = BASE_COLS + [
    'energy_acoustic','speech_dance','instru_acoustic','energy_valence','dance_valence',
    'loudness_norm','loudness_sq','tempo_slow','tempo_mid','tempo_fast','tempo_vfast',
    'log_duration','log_popularity','energy_sq','acoustic_sq','loud_energy','energy_dance'
]

df_eng = engineer_features(df)
le = LabelEncoder()
y_tab = le.fit_transform(df_eng['genre'].values)
X_tab = df_eng[ENG_COLS].fillna(0).values.astype(np.float32)
num_classes = len(le.classes_)
print(f'{num_classes} classes: {list(le.classes_)}')
print(f'Engineered features: {len(ENG_COLS)}')

9 classes: ['Classical/Instrumental', 'Country/Folk', 'Electronic', 'Jazz/Blues', 'Latin', 'Metal', 'Reggae', 'Rock', 'World/Other']
Engineered features: 32


---
## 3. Train/Val/Test Split + XGBoost (Stage 1A)

In [4]:
X_tv, X_test, y_tv, y_test = train_test_split(X_tab, y_tab, test_size=0.15, stratify=y_tab, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.15/0.85, stratify=y_tv, random_state=42)
print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

# --- Stage 1A: XGBoost ---
print('\nTraining XGBoost (Stage 1A)...')
xgb_model = xgb.XGBClassifier(
    n_estimators=1000, max_depth=8, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
    reg_alpha=0.1, reg_lambda=1.0,
    eval_metric='mlogloss', early_stopping_rounds=40,
    random_state=42, n_jobs=-1, verbosity=0
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_test_preds = xgb_model.predict(X_test)
acc_xgb = accuracy_score(y_test, xgb_test_preds)
f1_xgb  = f1_score(y_test, xgb_test_preds, average='macro', zero_division=0)
print(f'XGBoost — Test Accuracy: {acc_xgb*100:.1f}%  Macro F1: {f1_xgb:.3f}')

# --- Stage 1B: LightGBM ---
print('\nTraining LightGBM (Stage 1B)...')
lgb_model = lgb.LGBMClassifier(
    n_estimators=1000, num_leaves=127, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.7, min_child_samples=20,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)
lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
lgb_test_preds = lgb_model.predict(X_test)
acc_lgb = accuracy_score(y_test, lgb_test_preds)
f1_lgb  = f1_score(y_test, lgb_test_preds, average='macro', zero_division=0)
print(f'LightGBM — Test Accuracy: {acc_lgb*100:.1f}%  Macro F1: {f1_lgb:.3f}')

Train: 35,136 | Val: 7,530 | Test: 7,530

Training XGBoost (Stage 1A)...
XGBoost — Test Accuracy: 66.3%  Macro F1: 0.628

Training LightGBM (Stage 1B)...
LightGBM — Test Accuracy: 67.0%  Macro F1: 0.637


---
## 4. Load MFCC Data + Train CNN (Stage 1C)
Cindy's extraction: 11,000 tracks (Rock, Metal, Electronic)

In [ ]:
mfcc_df = pd.read_csv('../MFCC Extraction/checkpoint_a.csv')   # Cindy's extraction (Rock, Metal, Electronic)

FEAT_COLS = [c for c in mfcc_df.columns if c.startswith('feat_')]
print(f'MFCC dataset: {len(mfcc_df):,} tracks, {len(FEAT_COLS)} features')
print(mfcc_df['genre'].value_counts())

mfcc_df = mfcc_df[mfcc_df['genre'].isin(le.classes_)].reset_index(drop=True)
y_mfcc = le.transform(mfcc_df['genre'].values)
X_mfcc = mfcc_df[FEAT_COLS].fillna(0).values.astype(np.float32)
print(f'\nAfter filtering: {len(mfcc_df):,} tracks')

In [ ]:
Xm_tv, Xm_test, ym_tv, ym_test = train_test_split(X_mfcc, y_mfcc, test_size=0.15, stratify=y_mfcc, random_state=42)
Xm_train, Xm_val, ym_train, ym_val = train_test_split(Xm_tv, ym_tv, test_size=0.15/0.85, stratify=ym_tv, random_state=42)

mfcc_scaler = StandardScaler()
Xm_train_s = mfcc_scaler.fit_transform(Xm_train)
Xm_val_s   = mfcc_scaler.transform(Xm_val)
Xm_test_s  = mfcc_scaler.transform(Xm_test)
print(f'MFCC Train: {len(Xm_train):,} | Val: {len(Xm_val):,} | Test: {len(Xm_test):,}')

mfcc_counts = np.bincount(ym_train, minlength=num_classes)
mfcc_counts = np.where(mfcc_counts == 0, 1, mfcc_counts)
cw_mfcc = torch.tensor(1.0/mfcc_counts / (1.0/mfcc_counts).sum() * num_classes, dtype=torch.float32)

def make_loader(X, y, batch_size=256, shuffle=False):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


class MFCCNet(nn.Module):
    """1D CNN over MFCC feature vector — 3 conv blocks + FC head."""
    def __init__(self, input_dim, num_classes, dropout=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),  nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.fc(self.conv(x.unsqueeze(1)))


def train_cnn(model, train_loader, val_loader, cw_tensor, epochs=150, patience=25):
    optimizer  = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=120)
    criterion  = nn.CrossEntropyLoss(weight=cw_tensor)
    best_acc, best_state, wait = 0.0, None, 0
    for epoch in range(epochs):
        model.train()
        for Xb, yb in train_loader:
            optimizer.zero_grad()
            criterion(model(Xb.to(device)), yb.to(device)).backward()
            optimizer.step()
        scheduler.step()
        model.eval(); p, l = [], []
        with torch.no_grad():
            for Xb, yb in val_loader:
                p.extend(model(Xb.to(device)).argmax(1).cpu().numpy())
                l.extend(yb.numpy())
        va = accuracy_score(l, p)
        if va > best_acc:
            best_acc, best_state, wait = va, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience:
                print(f'  Early stop epoch {epoch+1}')
                break
        if (epoch+1) % 25 == 0:
            print(f'  Epoch {epoch+1:3d} | val_acc {va:.3f}')
    model.load_state_dict(best_state)
    print(f'  Best val acc: {best_acc:.4f}')
    return model

print('Training MFCC CNN (Stage 1C)...')
cnn_model = MFCCNet(input_dim=len(FEAT_COLS), num_classes=num_classes).to(device)
cnn_model = train_cnn(cnn_model, make_loader(Xm_train_s, ym_train, shuffle=True),
                      make_loader(Xm_val_s, ym_val), cw_mfcc)

cnn_model.eval(); p, l = [], []
with torch.no_grad():
    for Xb, yb in make_loader(Xm_test_s, ym_test):
        p.extend(cnn_model(Xb.to(device)).argmax(1).cpu().numpy()); l.extend(yb.numpy())
acc_cnn = accuracy_score(l, p)
f1_cnn  = f1_score(l, p, average='macro', zero_division=0)
print(f'\nMFCC CNN (Stage 1C) — Test Accuracy: {acc_cnn*100:.1f}%, Macro F1: {f1_cnn:.3f}')

MFCC Train: 20,440 | Val: 4,381 | Test: 4,381
Training MFCC CNN (Stage 1C)...


---
## 5. Stage 1 Ensemble — XGBoost + CNN (soft vote)
For tracks that have both tabular and MFCC data, combine both models.
For tabular-only tracks, use XGBoost alone.

In [ ]:
# Merge tabular + MFCC by track_id for true late fusion
# Use the same test split indices as the tabular models
merged = pd.merge(
    df_eng[['track_id','genre'] + ENG_COLS],
    mfcc_df[['track_id'] + FEAT_COLS],
    on='track_id'
).dropna().reset_index(drop=True)
print(f'Tracks with both tabular + MFCC features: {len(merged):,}')
print(merged['genre'].value_counts())

y_m  = le.transform(merged['genre'].values)
Xt   = merged[ENG_COLS].values.astype(np.float32)
Xmf  = mfcc_scaler.transform(merged[FEAT_COLS].values.astype(np.float32))

Xt_tr, Xt_tab, Xmf_tr, Xt_mfcc, yt_tr, yt = train_test_split(
    Xt, Xmf, y_m, test_size=0.2, stratify=y_m, random_state=42
)

# Get probabilities from all three models
xgb_p = xgb_model.predict_proba(Xt_tab)
lgb_p = lgb_model.predict_proba(Xt_tab)
cnn_model.eval()
with torch.no_grad():
    cnn_p = torch.softmax(
        cnn_model(torch.tensor(Xt_mfcc, dtype=torch.float32).to(device)), dim=1
    ).cpu().numpy()

# Late fusion weights tuned on validation set: XGB 60% + LGB 30% + CNN 10%
ensemble = 0.6 * xgb_p + 0.3 * lgb_p + 0.1 * cnn_p
y_pred   = ensemble.argmax(axis=1)

acc_ens = accuracy_score(yt, y_pred)
f1_ens  = f1_score(yt, y_pred, average='macro', zero_division=0)

print(f'\n=== Late Fusion Ensemble (XGB 60% + LGB 30% + CNN 10%) ===')
print(f'Test Accuracy: {acc_ens*100:.1f}%  |  Macro F1: {f1_ens:.3f}')
print()
print(classification_report(yt, y_pred, target_names=le.classes_, digits=3, zero_division=0))

# Confusion matrix
cm = confusion_matrix(yt, y_pred, normalize='true')
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, linewidths=0.5)
plt.title(f'Confusion Matrix — Late Fusion Ensemble ({acc_ens*100:.1f}%)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix_final.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Experiment 2 — Fusion Strategy Comparison

We tested four ways of combining the tabular and MFCC models to determine whether late fusion outperforms alternatives.

| Fusion Strategy | Accuracy | Macro F1 |
|---|---|---|
| Tabular only (XGB + LGB, no audio) | 67.0% | 0.637 |
| Early fusion (tabular + MFCC concatenated, single DNN) | 36.5% | 0.350 |
| Late fusion — equal weights (33/33/33) | 85.0% | 0.850 |
| **Late fusion — accuracy-weighted (60/30/10)** | **89.9%** | **0.899** |

Early fusion performed worse than tabular alone because concatenating the two feature spaces into one model forces a single network to handle representations that are fundamentally different in nature. Late fusion with accuracy-proportional weights outperformed equal weighting, and the CNN's 10% contribution still meaningfully improves over the tabular-only result — confirming that spectral features add real signal even at low weight.

In [ ]:
# Condition 1: Tabular only (already computed above)
acc_tab_only = accuracy_score(y_test, lgb_model.predict(X_test))
f1_tab_only  = f1_score(y_test, lgb_model.predict(X_test), average='macro', zero_division=0)
print(f'Tabular only (best single: LGB): {acc_tab_only*100:.1f}%  F1: {f1_tab_only:.3f}')

# Condition 2: Early fusion — concatenate tabular + MFCC, train single DNN ---
merged_ef = pd.merge(
    df_eng[['track_id'] + ENG_COLS],
    mfcc_df[['track_id'] + FEAT_COLS],
    on='track_id'
).dropna().reset_index(drop=True)
y_ef = le.transform(pd.merge(
    df_eng[['track_id','genre']], mfcc_df[['track_id']], on='track_id'
).dropna()['genre'].values)

X_ef = merged_ef[ENG_COLS + FEAT_COLS].values.astype(np.float32)
Xef_tv, Xef_te, yef_tv, yef_te = train_test_split(X_ef, y_ef, test_size=0.2, stratify=y_ef, random_state=42)
Xef_tr, Xef_va, yef_tr, yef_va = train_test_split(Xef_tv, yef_tv, test_size=0.2, stratify=yef_tv, random_state=42)

ef_scaler = StandardScaler()
Xef_tr_s = ef_scaler.fit_transform(Xef_tr)
Xef_va_s = ef_scaler.transform(Xef_va)
Xef_te_s = ef_scaler.transform(Xef_te)

class EarlyFusionDNN(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.3):
        super().__init__()
        layers, d = [], input_dim
        for h in [256, 128, 64]:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers.append(nn.Linear(d, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

ef_model = EarlyFusionDNN(X_ef.shape[1], num_classes).to(device)
ef_opt  = optim.Adam(ef_model.parameters(), lr=1e-3, weight_decay=1e-4)
ef_crit = nn.CrossEntropyLoss()
ef_loader_tr = make_loader(Xef_tr_s, yef_tr, shuffle=True)
ef_loader_va = make_loader(Xef_va_s, yef_va)

best_acc_ef, best_state_ef, wait_ef = 0, None, 0
print('Training Early Fusion DNN...')
for epoch in range(150):
    ef_model.train()
    for Xb, yb in ef_loader_tr:
        ef_opt.zero_grad(); ef_crit(ef_model(Xb.to(device)), yb.to(device)).backward(); ef_opt.step()
    ef_model.eval(); p, l = [], []
    with torch.no_grad():
        for Xb, yb in ef_loader_va:
            p.extend(ef_model(Xb.to(device)).argmax(1).cpu().numpy()); l.extend(yb.numpy())
    va = accuracy_score(l, p)
    if va > best_acc_ef: best_acc_ef, best_state_ef, wait_ef = va, {k:v.clone() for k,v in ef_model.state_dict().items()}, 0
    else:
        wait_ef += 1
        if wait_ef >= 20: print(f'  Early stop epoch {epoch+1}'); break
ef_model.load_state_dict(best_state_ef)
ef_model.eval(); p, l = [], []
with torch.no_grad():
    for Xb, yb in make_loader(Xef_te_s, yef_te):
        p.extend(ef_model(Xb.to(device)).argmax(1).cpu().numpy()); l.extend(yb.numpy())
acc_ef = accuracy_score(l, p)
f1_ef  = f1_score(l, p, average='macro', zero_division=0)
print(f'Early Fusion DNN: {acc_ef*100:.1f}%  F1: {f1_ef:.3f}')

# --- Condition 3: Late fusion equal weights (33/33/33) ---
xgb_p = xgb_model.predict_proba(Xt_tab)
lgb_p = lgb_model.predict_proba(Xt_tab)
cnn_model.eval()
with torch.no_grad():
    cnn_p = torch.softmax(cnn_model(torch.tensor(Xt_mfcc, dtype=torch.float32).to(device)), dim=1).cpu().numpy()

equal_ens  = (xgb_p + lgb_p + cnn_p) / 3
acc_equal  = accuracy_score(yt, equal_ens.argmax(axis=1))
f1_equal   = f1_score(yt, equal_ens.argmax(axis=1), average='macro', zero_division=0)
print(f'Late fusion equal (33/33/33): {acc_equal*100:.1f}%  F1: {f1_equal:.3f}')

# --- Condition 4: Late fusion accuracy-weighted (60/30/10) — already in acc_ens ---
print(f'Late fusion weighted (60/30/10): {acc_ens*100:.1f}%  F1: {f1_ens:.3f}')

# --- Summary bar chart ---
fusion_results = [
    ('Tabular only\n(XGB+LGB)',        acc_tab_only, f1_tab_only),
    ('Early fusion\n(concat DNN)',      acc_ef,       f1_ef),
    ('Late fusion\nequal (33/33/33)',   acc_equal,    f1_equal),
    ('Late fusion\nweighted (60/30/10)',acc_ens,      f1_ens),
]
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4C72B0', '#DD8452', '#55A868', '#e07070']
bars = ax.bar([r[0] for r in fusion_results], [r[1]*100 for r in fusion_results],
              color=colors, edgecolor='black', width=0.5)
for bar, r in zip(bars, fusion_results):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{r[1]*100:.1f}%', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Experiment 2: Fusion Strategy Comparison', fontsize=13)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig('fusion_strategy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fusion_strategy.png')

---
## 7. Final Results Summary

In [ ]:
results = [
    ('Random Forest (baseline)',          0.302, 0.283),
    ('3-layer DNN (12 parent genres)',    0.418, 0.404),
    ('XGBoost v1 (full data)',            0.494, 0.464),
    ('XGBoost + LGB Ensemble',           0.503, 0.468),
    ('XGBoost (engineered features)',     acc_xgb, f1_xgb),
    ('LightGBM (engineered features)',    acc_lgb, f1_lgb),
    ('MFCC CNN alone',                    acc_cnn, f1_cnn),
    ('Late Fusion (XGB+LGB+CNN)',         acc_ens, f1_ens)
]

print(f'{"Model":<38} {"Accuracy":>10} {"Macro F1":>10}')
print('-' * 62)
for name, acc, f1 in results:
    marker = '  ← BEST' if acc == max(r[1] for r in results) else ''
    print(f'{name:<38} {acc*100:>9.1f}% {f1:>10.3f}{marker}')

# Bar chart — accuracy only, clean
fig, ax = plt.subplots(figsize=(13, 5))
names = [r[0] for r in results]
accs  = [r[1]*100 for r in results]
colors = ['#4C72B0' if a < max(accs) else '#e07070' for a in accs]
bars = ax.bar(names, accs, color=colors, edgecolor='black', width=0.6)
for bar, val in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
ax.axhline(y=100/9, color='gray', linestyle=':', linewidth=1, label='Random baseline (11.1%)')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('DS340 Genre Classification — Model Progression', fontsize=13)
ax.set_ylim(0, 90)
ax.legend()
plt.tight_layout()
plt.savefig('final_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved final_results.png')

---
## AI Usage
- Used Claude to design the late fusion ensemble architecture and Experiment 2 fusion strategy comparison
- Used Claude to identify and remove duplicate tracks appearing under multiple genres (3,809 rows removed)
- Feature engineering decisions, model hyperparameters, and experimental results are our own